In [2]:
import edgar

from equibot.config import settings
from equibot.parsers import transcript
from etl.db import connect


edgar.set_identity(settings.sec_api_username)

with connect() as conn:                      # closes the connection on exit
    print(conn.execute("SELECT version()").fetchone()["version"])

PostgreSQL 18.6 (Debian 18.6-1.pgdg13+2) on x86_64-pc-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit


In [3]:
import laya

lagent = laya.load("convaiinnovations/laya-multilingual")

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]c:\Users\benak\repos\EquiBot\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\benak\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\benak\repos\EquiBot\.venv\Lib\site-packages\huggingface_hub\file_download.py:14

In [4]:
# get sections
snow = edgar.Company("SNOW")
tenq = snow.get_filings(form="10-Q")[0].obj()
sections = tenq.sections

In [71]:
filing = snow.get_filings(form="10-Q")[0]

In [ ]:
filing.obj()

'10-Q'

In [22]:
s = list(sections.keys())[3]
text = sections.get(s).text()  # financial statements

In [8]:
tokens = lagent.tok(text, return_tensors="pt")
tokens.input_ids.shape[1]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (32106 > 8192). Running this sequence through the model will result in indexing errors


32106

In [9]:
for s in sections:
    text = sections.get(s).text()
    tokens = lagent.tok(text, return_tensors="pt")
    print(f"Section: {s}, Token count: {tokens.input_ids.shape[1]}")

Section: part_i_item_1, Token count: 32106
Section: part_i_item_2, Token count: 20477
Section: part_i_item_3, Token count: 1507
Section: part_i_item_4, Token count: 537
Section: part_ii_item_1, Token count: 1379
Section: part_ii_item_1a, Token count: 30783
Section: part_ii_item_2, Token count: 266
Section: part_ii_item_3, Token count: 16
Section: part_ii_item_4, Token count: 15
Section: part_ii_item_5, Token count: 407
Section: part_ii_item_6, Token count: 712
Section: part_ii_signatures, Token count: 5


In [ ]:
# question sets by report type
choices = {
    "front_matter": "Front  Matter -- cover page, table of contents, forward-looking statements",
    "business": "Business -- company overview, products, competition, regulation, employees",
    "risk_factors": "Risk Factors -- risks facing the company",
    "cybersecurity": "Cybersecurity -- cyber risk management and governance",
    "properties": "Properties -- facilities and real estate",
    "legal_proceedings": "Legal Proceedings -- litigation and legal matters",
    "equity_and_repurchases": "Equity and Repurchases -- stock market info, dividends, buybacks",
    "mdna": "MD&A -- management's discussion of results and liquidity",
    "market_risk": "Market Risk -- interest rate, FX, commodity risk disclosures",
    "financial_statements": "Financial Statements -- balance sheet, income statement, cash flows",
    "notes_to_financial_statements": "Notes to Financial Statements -- accounting policies and note disclosures",
    "auditors_report": "Auditor's Report -- independent auditor's opinion",
    "controls_and_accountants": "Controls and Accountants -- internal controls, auditor changes",
    "governance_and_management": "Governance and Management -- directors, officers, related transactions",
    "compensation_and_ownership": "Compensation and Ownership -- executive pay, beneficial ownership",
    "exhibits_and_schedules": "Exhibits and Schedules -- exhibit index, financial statement schedules",
    # "signatures": "Signatures -- signature pages, power of attorney",
    "material_events": "Material Events -- significant events affecting the company; deals, financing, impairments, bankruptcy",
    "short_or_na_items": "Short or N/A Items -- 'None' or 'Not applicable' items",
    "other": "Other -- anything else",
}

questions = {
    "10-K": {'sections': {  # 'sections' is arbitrary, but laya expects a top-level key for the question set
        "type": "choice",
        "instructions": "Which part of the 10-K does this text belong to?",
        "criteria": {c: choices[c] for c in choices if c not in ('material_events')},
    }},
    "10-Q": {'sections': {
        "type": "choice",
        "instructions": "Which part of the 10-Q does this text belong to?",
        "criteria": {
            c: choices[c] for c in choices if c not in (
                'business', 'cybersecurity', 'properties', 'auditors_report', 
                'governance_and_management', 'compensation_and_ownership', 
                'material_events'
    )}}},
    "8-K": {'sections': {
        "type": "choice",
        "instructions": "Which part of the 8-K does this text belong to?",
        "criteria": {
            c: choices[c] for c in choices if c not in (
                'business', 'risk_factors', 'cybersecurity', 'properties', 
                'legal_proceedings', 'market_risk', 'notes_to_financial_statements', 
                'auditors_report', 'compensation_and_ownership', 'short_or_na_items'
    )}}}
}


In [28]:
from laya import Router

router = Router()  

In [83]:
def score_sections(filing, laya_instance, max_len: int | None = None):
    """Attribute each section in filing to a section label using Laya, 
    optionally truncating text to max_len characters
    """
    sections = filing.obj().sections
    results = {}
    for s in sections:
        # first few lines of a section typically reveal the category -- so truncation can be efficient
        if max_len is not None:
            text = sections.get(s).text()[:max_len]
        else:
            text = sections.get(s).text()
        res = laya_instance.predict(state=text, questions=questions[filing.form])  # limit choices relevant to filing's form type
        results[s] = {
            "pred": res["answers"]["sections"]["choice"],
            "confidence": res["answers"]["sections"]["answer_confidence"],
            "response": res,
            "header": text[:60],
            "text": text,
        }
    return results

In [84]:
results2 = score_sections(filing, router, max_len=100)
for s in results2:
    print(f"Section: {s}, Prediction: {results2[s]['pred']}, Confidence: {results2[s]['confidence']}")

print(f"Average Confidence: {sum(results2[s]['confidence'] for s in results2) / len(results2)}")

Section: part_i_item_1, Prediction: financial_statements, Confidence: 0.9994
Section: part_i_item_2, Prediction: mdna, Confidence: 0.92
Section: part_i_item_3, Prediction: market_risk, Confidence: 0.9651
Section: part_i_item_4, Prediction: other, Confidence: 0.2418
Section: part_ii_item_1, Prediction: legal_proceedings, Confidence: 1.0
Section: part_ii_item_1a, Prediction: risk_factors, Confidence: 1.0
Section: part_ii_item_2, Prediction: equity_and_repurchases, Confidence: 0.9997
Section: part_ii_item_3, Prediction: short_or_na_items, Confidence: 0.9747
Section: part_ii_item_4, Prediction: short_or_na_items, Confidence: 0.8399
Section: part_ii_item_5, Prediction: other, Confidence: 0.9472
Section: part_ii_item_6, Prediction: exhibits_and_schedules, Confidence: 1.0
Section: part_ii_signatures, Prediction: short_or_na_items, Confidence: 0.4833
Average Confidence: 0.8642583333333334


In [ ]:
filing_id = 1  # placeholder

def insert_section(filing, filing_id: int | str, section: str) -> int:
    """Insert section into raw.filing_section"""
    router = Router() 
    laya_res = score_sections(filing, router, max_len=100)
    section_pred = laya_res.get("pred")
    



    pass

'Table of Contents'

In [1]:
class Blah:
    def __init__(self):
        pass

    def do_something(self):
        print('yah')

    __call__ = do_something

In [7]:
Blah().__call__()

yah
